# insurance-trend

**Loss cost trend analysis for UK personal lines insurance pricing -- log-linear trend fitting with structural break detection, seasonal adjustment, and bootstrap confidence intervals.**

This notebook fits frequency and severity trends to synthetic UK motor data, detects the COVID lockdown structural break, decomposes the loss cost trend, and projects forward to the next rating period.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/insurance-trend/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q insurance-trend numpy

## 1. Synthetic quarterly experience data

24 quarters of UK motor aggregate data (2019 Q1 to 2024 Q4) with a known DGP:
- **Frequency:** stable at ~0.085 pre-2021, then a sharp drop (COVID lockdown at Q8), then recovery at +3% pa
- **Severity:** accelerates at +8% pa from 2022 (repair cost inflation)

The structural break is the critical test case. Naive OLS blends the pre- and post-break trends and produces a frequency estimate of approximately -8% pa -- badly wrong. The library detects the break and reports the correct post-break trend.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
n = 24  # 6 years of quarterly data

periods = [
    f"{yr}Q{q}"
    for yr in range(2019, 2025)
    for q in range(1, 5)
]

# Earned vehicle-years: growing book with slight noise
t = np.arange(n)
earned_vehicles = (12_000 + t * 120 + rng.normal(0, 180, n)).clip(10_000, None)

# True frequency: stable pre-2021 (Q0-Q7), COVID drop at Q8, recovery at +3% pa
freq_true = np.where(
    t < 8,
    0.085 + 0.001 * t,
    0.062 * np.exp(0.0074 * (t - 8)),   # +3% pa recovery (quarterly: exp(log(1.03)/4))
)
claim_counts = rng.poisson(freq_true * earned_vehicles).astype(float)

# True severity: modest pre-2022 (Q0-Q11), then +8% pa repair inflation
base_severity = 3_600.0
sev_true = base_severity * np.where(
    t < 12,
    1.0 + 0.005 * t,
    (1.0 + 0.005 * 11) * np.exp(0.0192 * (t - 11)),  # +8% pa from Q12
)
total_paid = claim_counts * rng.lognormal(np.log(sev_true), 0.12)

# Observed frequency and severity for inspection
obs_freq = claim_counts / earned_vehicles
obs_sev = total_paid / claim_counts.clip(1, None)

print(f"Periods: {periods[0]} to {periods[-1]}")
print(f"Earned vehicle-years range: {earned_vehicles.min():.0f} to {earned_vehicles.max():.0f}")
print(f"Observed frequency range: {obs_freq.min():.4f} to {obs_freq.max():.4f}")
print(f"Observed severity range: £{obs_sev.min():.0f} to £{obs_sev.max():.0f}")
print(f"\nTrue post-break trends (what we expect to recover):")
print(f"  Frequency: +3.0% pa")
print(f"  Severity:  +8.0% pa")
print(f"  Loss cost: ~+11.2% pa")

## 2. Fit the combined loss cost trend

`LossCostTrendFitter` wraps both frequency and severity fitters. With `detect_breaks=True`, the PELT algorithm runs on the log-transformed series to identify structural breaks before fitting the trend.

In [ ]:
from insurance_trend import LossCostTrendFitter

fitter = LossCostTrendFitter(
    periods=periods,
    claim_counts=claim_counts,
    earned_exposure=earned_vehicles,
    total_paid=total_paid,
)

result = fitter.fit(
    detect_breaks=True,   # PELT break detection on log-transformed series
    seasonal=True,        # quarterly seasonal dummies
    n_bootstrap=500,      # bootstrap CI replicates (1000 is default; 500 for speed)
)

print(result.summary())

## 3. Decompose the trend

`decompose()` breaks the combined loss cost trend into frequency and severity components. This is what goes into the pricing actuarial report: the overall trend is communicated, but underwriters want to know whether it is driven by more claims or more expensive claims.

In [ ]:
decomp = result.decompose()
print("Loss cost trend decomposition:")
print(f"  Frequency trend:  {decomp['freq_trend']:+.2%} pa")
print(f"  Severity trend:   {decomp['sev_trend']:+.2%} pa")
print(f"  Combined trend:   {decomp['combined_trend']:+.2%} pa")
if decomp['superimposed'] is not None:
    print(f"  Superimposed:     {decomp['superimposed']:+.2%} pa (net of external index)")

print()
print("Comparison to known DGP:")
print(f"  Frequency: fitted {decomp['freq_trend']:+.2%} vs true +3.00%")
print(f"  Severity:  fitted {decomp['sev_trend']:+.2%} vs true +8.00%")
print(f"  Combined:  fitted {decomp['combined_trend']:+.2%} vs true +11.24%")

## 4. Structural break detection

The frequency break corresponds to the COVID lockdown at Q8 (2020 Q1). The library should detect it and refit on the post-break segment -- recovering the +3% pa recovery trend rather than the blended -8% pa that naive OLS would return.

In [ ]:
freq_result = result.frequency
sev_result = result.severity

print("Frequency fit:")
print(f"  Method:      {freq_result.method}")
print(f"  Trend:       {freq_result.trend_rate:+.2%} pa")
print(f"  95% CI:      ({freq_result.ci_lower:+.2%}, {freq_result.ci_upper:+.2%})")
print(f"  Changepoints detected at quarters: {freq_result.changepoints}")
print(f"  R-squared:   {freq_result.r_squared:.4f}")

print()
print("Severity fit:")
print(f"  Method:      {sev_result.method}")
print(f"  Trend:       {sev_result.trend_rate:+.2%} pa")
print(f"  95% CI:      ({sev_result.ci_lower:+.2%}, {sev_result.ci_upper:+.2%})")
print(f"  Changepoints detected at quarters: {sev_result.changepoints}")
print(f"  R-squared:   {sev_result.r_squared:.4f}")

if freq_result.changepoints:
    q = freq_result.changepoints[0]
    print(f"\nFrequency break at quarter index {q} = period {periods[q] if q < len(periods) else 'out of range'}")
    print("True break: Q8 = 2020 Q1 (COVID lockdown)")

## 5. Forward projection

`trend_factor(n_periods)` gives the compound trend factor for use in rate-on-rate calculations. For a 12-month (4 quarter) projection:

In [ ]:
# 4-quarter (12-month) compound trend factor
factor_12m = result.trend_factor(n_periods=4)
print(f"12-month compound loss cost trend factor: {factor_12m:.4f}")
print(f"Implied rate need (trend only): {(factor_12m - 1):+.2%}")

# View the projection DataFrame
print("\nCombined loss cost projection (next 4 quarters):")
print(result.projection)

## 6. Individual component fitters

You can also use `FrequencyTrendFitter` and `SeverityTrendFitter` directly when you want to inspect or configure each component separately.

In [ ]:
from insurance_trend import FrequencyTrendFitter

freq_fitter = FrequencyTrendFitter(
    periods=periods,
    claim_counts=claim_counts,
    earned_exposure=earned_vehicles,
)

# Impose the known break at Q8 rather than relying on auto-detection
freq_result_imposed = freq_fitter.fit(
    detect_breaks=False,
    changepoints=[8],      # impose COVID break at Q8 = 2020 Q1
    seasonal=True,
    n_bootstrap=200,
)

print("Frequency trend with imposed break at Q8:")
print(freq_result_imposed.summary())

## What you should see

- Combined loss cost trend close to the true +11.2% pa, with the frequency and severity components close to +3% pa and +8% pa respectively
- Structural break detected near Q8 for frequency (COVID lockdown), and near Q12 for severity (repair inflation inflection)
- The fitted trend rate reflects the post-break regime -- what you would project forward, not the blended pre/post average
- 12-month trend factor of approximately 1.11 (11% compound uplift)

Naive OLS on this data would return a frequency trend of roughly -8% pa because the 35% lockdown step-down drags the entire fitted line downward. That is the wrong number to use in pricing. The library's approach of detecting and discarding the pre-break segment is the defensible choice for projection.

## Next steps

- **`ExternalIndex.from_ons('HPTH')`** -- deflate severity against the ONS motor repair index; `superimposed_inflation` gives the residual trend not explained by general cost inflation
- **`method='local_linear_trend'`** -- Kalman filter approach for slowly-changing trends (longer series required)
- **`changepoints=[8, 20]`** -- impose known structural events rather than relying on auto-detection when you know the dates

**GitHub:** https://github.com/burning-cost/insurance-trend  
**PyPI:** https://pypi.org/project/insurance-trend/